# Fine-tune IFM/K2-Horizon-MoVA-36B-A4B (QLoRA)

36B-parameter MoE, about 4B active parameters per token, native context 524,288. Checkpoint is BF16. Architecture is `K2HorizonForCausalLM` (`model_type: k2_horizon`) with remote code: dense MLP on layers 0–2, MoE (100 experts, 8 active, 1 shared) plus MoVA value experts (64 experts, 4 active) on the rest.

Unsloth does not ship a Fast K2-Horizon patch, so this notebook does not get the Unsloth kernel speedup. It follows the same QLoRA flow as the Unsloth MoE notebooks: 4-bit load, LoRA on attention and expert projections, routers left frozen, TRL `SFTTrainer`, then upload the adapter to the Hub.

Several datasets are concatenated into one training run. `weight` is the fraction of each split to keep before the mix, so a large set does not drown a small one. Paste an HF token once; it is used for the model download, every dataset, and the final upload. `hf_transfer` is enabled for faster Hub downloads.

Needs an NVIDIA GPU with at least 40 GB VRAM for rank 8 and sequence length 2048 (A100 80 GB is the comfortable size). The BF16 checkpoint is about 72 GB on disk before 4-bit conversion.


### Installation


In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q "transformers==5.15.0" "accelerate>=1.6.0" "peft>=0.17.0" "trl>=0.22.0" "datasets>=3.0.0" "bitsandbytes>=0.46.0" "huggingface_hub>=0.34.0" hf_transfer


### Hugging Face token


In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

# Write-token with access to the datasets you list and to the repo you upload to.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
login(token=HF_TOKEN, add_to_git_credential=False)

# Where the finished LoRA adapter is pushed.
hub_repo = "your-user/k2-horizon-mova-36b-a4b-lora"


### Load


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "IFM/K2-Horizon-MoVA-36B-A4B"
max_seq_length = 2048  # native context is 524288; raise this only if VRAM allows

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
model.config.use_cache = False


### LoRA

Train `q_proj`, `k_proj`, `v_proj`, `o_proj`, SwiGLU `gate_proj` / `up_proj` / `down_proj`, and the MoVA `v_experts.*` linears. Leave `mlp.gate` and `v_router` frozen. Rank 8 matches the small expert hidden size (768).


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=r".*(q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|v_experts\.\d+)$|.*mlp\.(experts|shared_experts).*gate_proj$",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


### Data

Every entry in `DATASETS` is downloaded in this cell, turned into chat text with the checkpoint template, then concatenated and shuffled into one train/test split. Set `weight` below 1 to keep a random fraction of that split. Add a row for each extra dataset. Columns accepted: `messages`, `conversations`, or ShareGPT `from`/`value`.


In [ ]:
from datasets import concatenate_datasets, load_dataset

DATASETS = [
    {"name": "mlabonne/FineTome-100k-cleaned", "split": "train", "weight": 1.0},
    {"name": "HuggingFaceH4/ultrachat_200k", "split": "train_sft", "weight": 0.25},
]

ROLE_MAP = {
    "human": "user",
    "gpt": "assistant",
    "system": "system",
    "user": "user",
    "assistant": "assistant",
}

def to_messages(example):
    conv = example.get("conversations") or example.get("messages")
    messages = []
    for turn in conv:
        role = turn.get("role") or turn.get("from")
        content = turn.get("content") or turn.get("value") or ""
        messages.append({"role": ROLE_MAP.get(role, role), "content": content})
    return {"messages": messages}

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                chat_template_kwargs={"reasoning_effort": "high"},
            )
        except TypeError:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

parts = []
for spec in DATASETS:
    split = load_dataset(spec["name"], split=spec["split"], token=HF_TOKEN)
    weight = float(spec.get("weight", 1.0))
    if weight < 1.0:
        split = split.shuffle(seed=3407).select(range(int(len(split) * weight)))
    split = split.map(to_messages, remove_columns=split.column_names)
    split = split.map(formatting_prompts_func, batched=True)
    print(f"{spec['name']} [{spec['split']}] -> {len(split)} rows (weight {weight})")
    parts.append(split)

dataset = concatenate_datasets(parts).shuffle(seed=3407)
dataset = dataset.train_test_split(test_size=0.01, seed=3407)
print(f"combined train={len(dataset['train'])} test={len(dataset['test'])}")


### Train


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        packing=False,
    ),
)
trainer.train()


### Save and upload

The adapter and tokenizer are written locally, then pushed to `hub_repo` with the same token.


In [ ]:
output_dir = "k2-horizon-mova-36b-a4b-lora"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

model.push_to_hub(hub_repo, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(hub_repo, token=HF_TOKEN, private=True)
print(f"https://huggingface.co/{hub_repo}")


### Inference


In [ ]:
model.gradient_checkpointing_disable()
model.eval()

messages = [{"role": "user", "content": "Explain why long-context evaluation is difficult."}]
try:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"reasoning_effort": "high"},
    )
except TypeError:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
inputs.pop("token_type_ids", None)
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=0.95,
        do_sample=True,
    )
print(tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
